# 02 — Обучение ResNet18 (ImageNet) на GTZAN

**Что делает ноутбук:**
- Запускает 5-fold × 3 seeds = **15 runs** для ResNet18 с adaptation `avg_conv1` (primary).
- Дополнительно: **3 channel-strategy ablation** на fold=0, seed=42:
    - `avg_conv1` (уже посчитан в основном цикле)
    - `replicate` (1ch→3ch repeat)
    - `reinit_conv1` (Kaiming init)
- Тот же two-stage pipeline, что и для CNN14 (изолируем эффект backbone).

**Выходы (`/kaggle/working/outputs/`):**
- `best_resnet18_fold{F}_seed{S}.pth` × 15 (+ ablation чекпойнты).
- `resnet18_metrics.csv`, `resnet18_history.csv`.
- `resnet18_ablation.csv` — сравнение 3 channel strategies.

**Время на T4:** ~3-5 часов на 15 runs + 2 ablation.

In [ ]:
!pip install -q librosa==0.10.1 h5py soxr fvcore grad-cam umap-learn 2>&1 | tail -3

In [ ]:
import sys
from pathlib import Path
for p in ['/kaggle/input/gtzan-cnn14-resnet18-src', '/kaggle/input/cnn14-resnet18-src',
          '/kaggle/working', str(Path.cwd().parent)]:
    if Path(p, 'src', '__init__.py').exists():
        sys.path.insert(0, p)
        break
import torch
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
def find_file(*candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return None

H5_PATH = find_file('/kaggle/input/gtzan-preproc/gtzan_logmel.h5',
                    '/kaggle/working/gtzan_logmel.h5',
                    Path.cwd().parent / 'outputs' / 'gtzan_logmel.h5')
FOLDS_PATH = find_file('/kaggle/input/gtzan-preproc/folds.json',
                       '/kaggle/working/folds.json',
                       Path.cwd().parent / 'outputs' / 'folds.json')
assert H5_PATH and FOLDS_PATH, 'Сначала запустите 00_data_prep'
OUT_DIR = Path('/kaggle/working/outputs')
OUT_DIR.mkdir(exist_ok=True, parents=True)
print(f'H5: {H5_PATH}, Folds: {FOLDS_PATH}')

In [ ]:
import time
import pandas as pd

from src.configs import TrainConfig, AugConfig, DEFAULT_SEEDS, N_FOLDS
from src.dataset import load_folds
from src.models import build_model
from src.train import fit_two_stage, build_loaders

folds = load_folds(FOLDS_PATH)

def make_cfg(fold: int, seed: int, channel_strategy: str = 'avg_conv1') -> TrainConfig:
    return TrainConfig(
        model_name='resnet18',
        channel_strategy=channel_strategy,
        fold=fold,
        seed=seed,
        run_name=f'resnet18_{channel_strategy}_f{fold}_s{seed}',
        batch_size=32,
        stage1_epochs=10,
        stage2_epochs=30,
        early_stop_patience=7,
        lr_backbone=1e-4,
        lr_head=1e-3,
        wd_backbone=1e-4,
        wd_head=1e-3,
        warmup_epochs=2,
        label_smoothing=0.1,
        grad_clip_norm=1.0,
        use_amp=True,
        aug=AugConfig(use_specaugment=True, use_mixup=True, mixup_alpha=0.2),
        h5_path=str(H5_PATH),
        folds_json_path=str(FOLDS_PATH),
        output_dir=str(OUT_DIR),
        num_workers=2,
    )

print(f'Запланировано: {N_FOLDS} × {len(DEFAULT_SEEDS)} = {N_FOLDS * len(DEFAULT_SEEDS)} runs + 2 ablation')

In [ ]:
# === ОСНОВНОЙ ЦИКЛ: ResNet18 / avg_conv1 на всех folds × seeds ===
all_results = []
all_histories = []
start = time.time()

for fold in range(N_FOLDS):
    fold_data = folds[str(fold)]
    train_ids, val_ids = fold_data['train'], fold_data['val']
    print(f'\n========== FOLD {fold} ==========')
    for seed in DEFAULT_SEEDS:
        cfg = make_cfg(fold, seed, 'avg_conv1')
        run_dir = OUT_DIR / 'runs' / cfg.run_name
        best_path = OUT_DIR / f'best_resnet18_fold{fold}_seed{seed}.pth'
        if best_path.exists():
            ckpt = torch.load(best_path, map_location='cpu')
            if ckpt.get('stage', 0) == 2:
                print(f'[skip] {cfg.run_name} (val_f1={ckpt["val_f1"]:.3f})')
                all_results.append({'fold': fold, 'seed': seed,
                                    'channel_strategy': 'avg_conv1',
                                    'best_val_f1': ckpt['val_f1']})
                continue

        print(f'\n>>> {cfg.run_name}')
        model = build_model('resnet18', channel_strategy='avg_conv1')
        train_loader, val_loader = build_loaders(str(H5_PATH), train_ids, val_ids, cfg)
        result = fit_two_stage(model, train_loader, val_loader, cfg, out_dir=OUT_DIR, tb_dir=run_dir)

        # Переименуем чекпойнт под общий шаблон (без channel_strategy в имени для primary)
        old_best = OUT_DIR / f'best_resnet18_fold{fold}_seed{seed}.pth'
        # fit_two_stage уже использует это имя

        for row in result['history']:
            all_histories.append({'model': 'resnet18', 'channel_strategy': 'avg_conv1',
                                  'fold': fold, 'seed': seed, **row})
        all_results.append({
            'fold': fold, 'seed': seed, 'channel_strategy': 'avg_conv1',
            'best_val_f1': result['best_val_f1'],
            'best_epoch': result['best_epoch'],
            'best_path': result['best_path'],
        })
        pd.DataFrame(all_histories).to_csv(OUT_DIR / 'resnet18_history.csv', index=False)
        pd.DataFrame(all_results).to_csv(OUT_DIR / 'resnet18_metrics.csv', index=False)
        del model, train_loader, val_loader
        torch.cuda.empty_cache()
        print(f'[total elapsed] {(time.time()-start)/3600:.2f}h')

In [ ]:
# === ABLATION: 3 channel strategies на fold=0, seed=42 ===
ablation_results = []
ablation_strategies = ['avg_conv1', 'replicate', 'reinit_conv1']
fold, seed = 0, 42
fold_data = folds[str(fold)]
train_ids, val_ids = fold_data['train'], fold_data['val']

for strat in ablation_strategies:
    print(f'\n>>> ablation: {strat}')
    cfg = make_cfg(fold, seed, strat)
    # avg_conv1 уже есть из основного цикла
    if strat == 'avg_conv1':
        best_path = OUT_DIR / f'best_resnet18_fold{fold}_seed{seed}.pth'
        if best_path.exists():
            ckpt = torch.load(best_path, map_location='cpu')
            ablation_results.append({'channel_strategy': strat, 'best_val_f1': ckpt['val_f1']})
            continue
    # Для replicate/reinit — отдельный путь чекпойнта
    abl_ckpt = OUT_DIR / f'ablation_resnet18_{strat}_fold{fold}_seed{seed}.pth'
    if abl_ckpt.exists():
        ckpt = torch.load(abl_ckpt, map_location='cpu')
        ablation_results.append({'channel_strategy': strat, 'best_val_f1': ckpt['val_f1']})
        continue

    model = build_model('resnet18', channel_strategy=strat)
    train_loader, val_loader = build_loaders(str(H5_PATH), train_ids, val_ids, cfg)
    # Для ablation сохраним в другое имя — переопределим run_name
    cfg.model_name = f'resnet18_{strat}'  # это меняет имя best_*.pth внутри fit_two_stage
    result = fit_two_stage(model, train_loader, val_loader, cfg, out_dir=OUT_DIR, tb_dir=None)
    # Переименуем под единое имя
    src_p = Path(result['best_path'])
    src_p.rename(abl_ckpt)
    ablation_results.append({'channel_strategy': strat, 'best_val_f1': result['best_val_f1']})
    del model, train_loader, val_loader
    torch.cuda.empty_cache()

abl_df = pd.DataFrame(ablation_results)
abl_df.to_csv(OUT_DIR / 'resnet18_ablation.csv', index=False)
print(abl_df.to_string(index=False))

In [ ]:
# Сводка ResNet18
df = pd.DataFrame(all_results)
print(df.to_string(index=False))
if not df.empty:
    print(f'\nResNet18 (avg_conv1) best val_f1: mean={df["best_val_f1"].mean():.3f} ± {df["best_val_f1"].std():.3f}')

# Cleanup
for p in OUT_DIR.glob('best_resnet18*.pth'):
    ckpt = torch.load(p, map_location='cpu')
    if 'optimizer' in ckpt:
        del ckpt['optimizer']
        torch.save(ckpt, p)